# 07b — Xếp hạng lại

Đây là bước tăng điểm lớn nhất của dự án, và là bước đầu tiên cần GPU.

### Vì sao bước này đáng làm

Notebook `06b` mục 9 chia 1.211 câu truy vấn theo *cần gì để sửa*:

| Nhóm | Số câu | Tỷ lệ |
| --- | --- | --- |
| Đã trúng trong top 10 | 571 | 47,2% |
| Đáp án nằm trong 100 ứng viên, chưa lên top | 361 | **29,8%** |
| Đáp án không có trong 100 ứng viên | 279 | **23,0%** |

Nhóm giữa là phần bộ xếp hạng lại giành được. Nhóm cuối thì không, vì xếp hạng
lại chỉ sắp xếp lại những gì tầng một đưa cho.

Nên notebook này làm **hai việc**, không phải một:

1. Nâng độ sâu để nhóm cuối co lại. `03b` đo được `recall@1000` là 0,6839 so
   với `recall@100` là 0,4075, và trần xếp hạng lại tăng từ 0,4584 lên 0,7296.
2. Xếp hạng lại danh sách sâu hơn đó bằng cross-encoder.

### Hai tầng khác nhau ở chỗ nào

| | Tầng một (`bm25`) | Xếp hạng lại (cross-encoder) |
| --- | --- | --- |
| Nhìn thấy gì | Từ trùng nhau | Cả câu truy vấn và cả tài liệu cùng lúc |
| Chi phí | Toàn kho, vài phút | Từng cặp, nên phải giới hạn số cặp |
| Việc làm được | Thu 1,65 triệu xuống 1000 | Sắp lại 1000 đó cho đúng |

Cross-encoder đọc câu truy vấn và tài liệu **cùng trong một lượt**, nên nó bắt
được quan hệ giữa chữ ở hai bên. Đó là thứ `bm25` không làm được vì `bm25` chỉ
cộng trọng số của các từ trùng nhau. Đổi lại, nó phải chạy một lượt cho mỗi
cặp, nên không thể áp lên cả kho.

### Chạy trên hai máy

Phần GPU nằm ở máy khác. Khung ở `06b` đã tính trước việc này:

```text
máy CPU                        máy GPU
--------                       --------
lấy ứng viên, khử trùng lặp
xuất gói việc          ──────► chạy gpu_worker.py
                               trả về scores.jsonl
nhập điểm              ◄──────
xếp hạng lại, chấm điểm
```

Gói việc chỉ chứa danh sách mã tài liệu và câu truy vấn, khoảng vài chục MB.
Máy GPU tự đọc kho văn bản. Nếu máy GPU không có kho thì xuất kèm luôn văn bản,
gói sẽ nặng hơn nhiều.

### Đầu ra

| Kết quả | Dùng ở đâu |
| --- | --- |
| Hệ thống có chặng `rerank` | `07d`, `08` |
| Phần trần đã giành được | Quyết định có cần `07c` không |
| Gói việc GPU tái lập được | Người thứ hai chạy lại |

In [ ]:
import sys
import json
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path("..") / "src"))
import reteco as R
import pipeline as P

DATA = Path(r"D:\RETECO-project\reteco_data\track1_tempo")
SYSTEMS = Path("..") / "systems"
GPU_JOBS = Path("..") / "gpu_jobs"
CACHE = Path("results")
GPU_JOBS.mkdir(exist_ok=True)

P.configure(DATA, CACHE)
domains = P.domains()

dedup_file = CACHE / "dedup_summary.json"
assert dedup_file.exists(), (
    "results/dedup_summary.json is missing. Run notebook 07a first: it "
    "chooses the dedup setting and the depth this notebook retrieves at.")
dedup_summary = json.loads(dedup_file.read_text(encoding="utf-8"))

print(f"{len(domains)} domains")
print(f"from 07a: depth {dedup_summary['rerank_depth']:,}, "
      f"expand {dedup_summary['chosen_expand']}")
print(f"rerank kinds available: {sorted(P._REGISTRY['rerank'])}")

In [ ]:
# --- Shared chart style -------------------------------------------------
# Same palette and helpers as every other notebook in the project, so a bar
# here means what a bar there means.
BLUE, ORANGE, TEAL, AMBER, PINK, VIOLET = (
    "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#4a3aa7")
SURFACE, INK, INK_SOFT, INK_MUTED = "#fcfcfb", "#0b0b0b", "#52514e", "#898781"
GRID, AXIS = "#e1e0d9", "#c3c2b7"

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "savefig.facecolor": SURFACE, "axes.edgecolor": AXIS,
    "axes.labelcolor": INK_SOFT, "axes.labelsize": 9.5,
    "text.color": INK, "xtick.color": INK_MUTED, "ytick.color": INK_MUTED,
    "xtick.labelsize": 9.5, "ytick.labelsize": 9.5, "font.size": 10,
    "figure.dpi": 120, "axes.linewidth": 0.9,
})


def finish(ax, title, subtitle=None, xlabel=None, ylabel=None,
           grid_axis="y", note=None):
    if subtitle:
        ax.set_title(subtitle, loc="left", pad=8, fontsize=9.5, color=INK_MUTED)
        ax.annotate(title, xy=(0, 1), xycoords="axes fraction",
                    xytext=(0, 24), textcoords="offset points",
                    ha="left", va="bottom", fontsize=12.5,
                    fontweight="bold", color=INK, annotation_clip=False)
    else:
        ax.set_title(title, loc="left", pad=12, fontsize=12.5,
                     fontweight="bold", color=INK)
    ax.set_xlabel(xlabel or "")
    ax.set_ylabel(ylabel or "")
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.spines["left" if grid_axis == "x" else "bottom"].set_color(AXIS)
    if grid_axis == "x":
        ax.spines["bottom"].set_visible(False)
    elif grid_axis:
        ax.spines["left"].set_visible(False)
    if grid_axis:
        ax.grid(axis=grid_axis, color=GRID, linewidth=0.8)
        ax.set_axisbelow(True)
    ax.tick_params(length=0)
    if note:
        ax.text(0, -0.34, note, transform=ax.transAxes, ha="left", va="top",
                fontsize=8.5, color=INK_MUTED)
    return ax


def label_bars(ax, bars, values, fmt="{:,.0f}", horizontal=True, pad=0.015):
    span = max(values) if len(values) else 1
    for bar, value in zip(bars, values):
        if horizontal:
            ax.text(bar.get_width() + span * pad,
                    bar.get_y() + bar.get_height() / 2, fmt.format(value),
                    va="center", ha="left", fontsize=8.5, color=INK_SOFT)
        else:
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + span * pad, fmt.format(value),
                    ha="center", va="bottom", fontsize=8.5, color=INK_SOFT)


def legend_below(ax, ncol=2, y=-0.30):
    ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, y),
              ncol=ncol, fontsize=9.5, handlelength=1.1, handleheight=1.1,
              columnspacing=1.8)

---

## 1. Tầng một, lấy sâu

Notebook `07a` đã ghi sẵn file cấu hình. Chặng này lấy về sâu rồi gộp bản sao,
để bộ xếp hạng lại nhận được nhiều nội dung khác nhau nhất trong cùng một ngân
sách GPU.

Lấy sâu gần như không tốn thêm: việc đắt là dựng chỉ mục, và nó chạy một lần
bất kể lấy 100 hay 2000. Chỉ có kích thước lát cắt là đổi.

In [ ]:
# --- Retrieve deep, then collapse copies ---------------------------------
stage1 = P.load_system(SYSTEMS / "03_stage1_deep.json")
print(json.dumps(stage1, indent=1))
print()

started = time.time()
stage1_digest = P.run(stage1, split="train")
P.score(stage1_digest)
print(f"\ntook {time.time() - started:.0f}s")

# The shallow system from 06b, for comparison.
shallow = P.load_system(SYSTEMS / "01_bm25_tuned.json")
shallow_digest = P.run(shallow, split="train")
P.score(shallow_digest)

record_deep = P.score(stage1_digest)
record_shallow = P.score(shallow_digest)

print(f"\n{'system':<20}{'nDCG@10':>10}{'R@100':>9}{'ceiling':>10}{'topics':>9}")
print("-" * 58)
for label, record in (("depth 100", record_shallow),
                      ("deep + dedup", record_deep)):
    print(f"{label:<20}{record['macro']['ndcg']:>10.4f}"
          f"{record['macro']['recall']:>9.4f}"
          f"{record['macro']['ceiling']:>10.4f}{record['n_topics']:>9}")
print("-" * 58)
print()
print("nDCG@10 barely moves: the top 10 is nearly the same list. What moved")
print("is `ceiling`, and that is the number this notebook spends GPU time to")
print("collect.")

In [ ]:
# --- How much material does the reranker actually get --------------------
deep_run = P._load_run(stage1_digest)
lengths = [len(hits) for per_query in deep_run.values()
           for hits in per_query.values()]
total_pairs = sum(lengths)

print(f"{'domain':<12}{'queries':>9}{'mean list':>11}{'shortest':>10}"
      f"{'longest':>9}")
print("-" * 51)
for domain in domains:
    sizes = [len(h) for h in deep_run.get(domain, {}).values()]
    if not sizes:
        continue
    print(f"{domain:<12}{len(sizes):>9}{np.mean(sizes):>11.0f}"
          f"{min(sizes):>10}{max(sizes):>9}")
print("-" * 51)
print(f"{'all':<12}{len(lengths):>9}{np.mean(lengths):>11.0f}"
      f"{min(lengths):>10}{max(lengths):>9}")
print()
print(f"Total pairs to score on the GPU: {total_pairs:,}")

# Rough bill. A small cross-encoder on a mid-range GPU does on the order of
# 700 pairs a second; the notebook prints a range rather than one number
# because it depends on the card and on how long the documents are.
for rate in (300, 700, 1500):
    print(f"  at {rate:>5,} pairs/s -> {total_pairs / rate / 60:>5.0f} minutes")

---

## 2. Trần mới

Trần xếp hạng lại là điểm cao nhất mà một bộ xếp hạng lại hoàn hảo đạt được
trên đúng danh sách ứng viên này. Nó là chặn trên, không phải ước lượng.

Biết trần trước khi chạy GPU trả lời được câu hỏi *bước này có đáng làm không*,
và sau khi chạy thì nó cho biết bộ xếp hạng lại **giành được bao nhiêu phần**
của khoảng trống đó.

In [ ]:
# --- What the deeper list is worth ---------------------------------------
TARGET = 0.30           # the score that would put the project in contention

points = [
    ("hien tai, do sau 100", record_shallow["macro"]["ndcg"],
     record_shallow["macro"]["ceiling"]),
    ("sau khi lay sau + gop", record_deep["macro"]["ndcg"],
     record_deep["macro"]["ceiling"]),
]

print(f"{'stage':<26}{'nDCG@10':>10}{'ceiling':>10}{'headroom':>11}"
      f"{'share of it for 0.30':>22}")
print("-" * 79)
for label, now, ceiling in points:
    headroom = ceiling - now
    share = (TARGET - now) / headroom if headroom > 0 else float("nan")
    print(f"{label:<26}{now:>10.4f}{ceiling:>10.4f}{headroom:>11.4f}"
          f"{share:>21.1%}")
print("-" * 79)
print()
print("The last column is the question this notebook answers: to reach")
print(f"{TARGET}, what share of the gap between today's score and the ceiling")
print("must the reranker collect. A deeper list lowers that share, which is")
print("why depth comes before reranking.")

In [ ]:
# --- Who is still out of reach at this depth -----------------------------
print("At depth 100 (measured in 06b):")
print(f"{'scored':<16}{571:>7}{0.472:>9.1%}")
print(f"{'recoverable':<16}{361:>7}{0.298:>9.1%}")
print(f"{'unreachable':<16}{279:>7}{0.230:>9.1%}")
print()
print("At the depth this notebook uses:")
buckets_deep = P.failure_breakdown(stage1_digest)

recoverable = len(buckets_deep["recoverable"])
unreachable = len(buckets_deep["unreachable"])
total_q = sum(len(v) for v in buckets_deep.values())
print()
print(f"The reranker can now touch {recoverable:,} queries "
      f"({recoverable / total_q:.1%}).")
print(f"{unreachable:,} ({unreachable / total_q:.1%}) stay out of reach even "
      f"at this depth;")
print("those are what notebook 07c tries to reach with a different kind of")
print("retrieval.")

In [ ]:
# --- The two depths side by side -----------------------------------------
fig, ax = plt.subplots(figsize=(9.4, 2.9))

rows = [
    ("do sau 100", 0.472, 0.298, 0.230),
    ("do sau sau hon", len(buckets_deep["scored"]) / total_q,
     recoverable / total_q, unreachable / total_q),
]
labels = ["Da trung top 10", "Xep hang lai cham toi duoc",
          "Ngoai tam voi"]
colours = [TEAL, AMBER, ORANGE]

for y, (name, *shares) in enumerate(rows):
    left = 0.0
    for share, colour in zip(shares, colours):
        ax.barh([y], [share], left=left, color=colour, height=0.5,
                label=labels[colours.index(colour)] if y == 0 else None)
        if share > 0.07:
            ax.text(left + share / 2, y, f"{share:.0%}", ha="center",
                    va="center", fontsize=9.5, color=SURFACE,
                    fontweight="bold")
        left += share

ax.set_yticks(range(len(rows)))
ax.set_yticklabels([r[0] for r in rows])
ax.invert_yaxis()
ax.set_xlim(0, 1)
ax.set_xticks([0, 0.25, 0.5, 0.75, 1.0])
ax.xaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
legend_below(ax, ncol=3, y=-0.30)
finish(ax, f"Lay sau hon keo nhom ngoai tam voi tu 23% xuong "
           f"{unreachable / total_q:.0%}",
       subtitle="Phan mau cam la phan khong bo xep hang lai nao cuu duoc",
       grid_axis=None)
plt.show()

---

## 3. Xuất gói việc cho máy GPU

Gói việc là một thư mục chứa danh sách ứng viên và câu truy vấn. Không chứa
mã của dự án, nên máy GPU chỉ cần `gpu_worker.py` và thư mục này.

Hai lựa chọn:

| `with_text` | Gói nặng bao nhiêu | Máy GPU cần gì |
| --- | --- | --- |
| `False` | vài chục MB | Có sẵn kho `track1_tempo` |
| `True` | hàng trăm MB | Không cần gì thêm |

Kho tải được từ HuggingFace bằng một lệnh, nên `False` thường tiện hơn.

In [ ]:
# --- Write the job folder -------------------------------------------------
JOB_ID = "rerank_01"
RERANK_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
WITH_TEXT = False        # True if the GPU machine has no copy of the corpus

job = P.export_gpu_job(
    stage1_digest, GPU_JOBS / JOB_ID, JOB_ID,
    task="rerank", split="train", with_text=WITH_TEXT, model=RERANK_MODEL)

folder = GPU_JOBS / JOB_ID
size = sum(p.stat().st_size for p in folder.iterdir()) / 1e6
print(f"\nfolder size: {size:.0f} MB")
for path in sorted(folder.iterdir()):
    print(f"  {path.name:<22}{path.stat().st_size / 1e6:>8.1f} MB")

### Lệnh chạy trên máy GPU

Chép thư mục `gpu_jobs/rerank_01/` và file `src/gpu_worker.py` sang máy có GPU,
rồi chạy:

```bash
pip install torch sentence-transformers

# neu may do chua co kho van ban
pip install huggingface_hub
hf download DataScience-UIBK/RETECO-SemEval2027 --repo-type dataset \
    --local-dir reteco_data --include "track1_tempo/*"

python gpu_worker.py rerank_01 --data reteco_data/track1_tempo
```

Nếu gói đã xuất kèm văn bản thì bỏ `--data` đi.

Công cụ ghi ra `rerank_01/scores.jsonl`, và ghi theo từng nhóm một. Phiên làm
việc bị ngắt giữa chừng thì chạy lại chính lệnh đó, nó đọc phần đã xong và làm
tiếp từ chỗ dừng.

Chép `scores.jsonl` về rồi chạy tiếp ô lệnh bên dưới.

In [ ]:
# --- Take the scores back ------------------------------------------------
scores_path = GPU_JOBS / JOB_ID / "scores.jsonl"

if not scores_path.exists():
    print(f"{scores_path} is not here yet.")
    print()
    print("Run the worker on the GPU machine, copy scores.jsonl back into")
    print(f"{GPU_JOBS / JOB_ID}, then run this cell again. Everything above")
    print("is cached, so nothing is recomputed.")
else:
    P.import_gpu_scores(scores_path, JOB_ID)
    print("Imported. The rerank stage can now read it.")

---

## 4. Xếp hạng lại và chấm điểm

Chặng `rerank` đọc file điểm vừa nhập và sắp lại từng danh sách. Tài liệu nào
máy GPU chưa chấm thì giữ nguyên chỗ ở cuối danh sách, nên một gói việc chạy dở
vẫn dùng được, chỉ là kém hơn.

In [ ]:
# --- Rerank, at a few cut depths -----------------------------------------
# The reranker reorders the whole list; the cut only decides how much of that
# order is kept. Cutting shallow is cheaper to store and makes no difference
# to nDCG@10, so this checks rather than assumes.
CUTS = [100, 1000]
rerank_digests = {}

if scores_path.exists():
    for cut in CUTS:
        system = dict(stage1,
                      name=f"rerank_{JOB_ID}_cut{cut}",
                      note=f"{RERANK_MODEL} over {stage1['name']}",
                      rerank={"kind": "gpu_job", "job": JOB_ID, "depth": cut})
        (SYSTEMS / f"04_rerank_cut{cut}.json").write_text(
            json.dumps(system, indent=2), encoding="utf-8")
        digest = P.run(system, split="train")
        P.score(digest)
        rerank_digests[f"rerank cut {cut}"] = digest

    print(f"\n{'system':<22}{'fit':>9}{'check':>9}{'all train':>11}"
          f"{'MAP':>9}{'ceiling':>10}{'topics':>9}")
    print("-" * 79)
    rows = [("stage 1 only", stage1_digest)] + list(rerank_digests.items())
    for label, digest in rows:
        r = P.score(digest)
        print(f"{label:<22}{r['fit']['ndcg']:>9.4f}{r['check']['ndcg']:>9.4f}"
              f"{r['macro']['ndcg']:>11.4f}{r['macro']['map']:>9.4f}"
              f"{r['macro']['ceiling']:>10.4f}{r['n_topics']:>9}")
    print("-" * 79)
else:
    print("Waiting for scores.jsonl. Section 3 says how to produce it.")

In [ ]:
# --- How much of the headroom did it collect -----------------------------
if rerank_digests:
    best_name = max(rerank_digests,
                    key=lambda n: P.score(rerank_digests[n])["fit"]["ndcg"])
    best_digest = rerank_digests[best_name]
    after = P.score(best_digest)

    before_score = record_deep["macro"]["ndcg"]
    ceiling = record_deep["macro"]["ceiling"]
    after_score = after["macro"]["ndcg"]
    collected = (after_score - before_score) / (ceiling - before_score)

    print(f"first stage        {before_score:.4f}")
    print(f"after reranking    {after_score:.4f}   "
          f"({after_score - before_score:+.4f})")
    print(f"perfect reranker   {ceiling:.4f}")
    print()
    print(f"The reranker collected {collected:.1%} of the gap between the")
    print("first stage and a perfect reordering of the same list.")
    print()
    print(f"{'comparison':<34}{'difference':>12}{'95% interval':>26}"
          f"{'verdict':>9}")
    print("-" * 81)
    rerank_significance = P.compare(best_digest, stage1_digest, part="fit")
    print("-" * 81)
else:
    collected = None
    rerank_significance = None

In [ ]:
# --- Per domain: where reranking helps, and where it does not ------------
if rerank_digests:
    before = P.score(stage1_digest)["per_domain"]
    after_domains = P.score(best_digest)["per_domain"]
    deltas = sorted(((d, after_domains[d]["ndcg"] - before[d]["ndcg"],
                      before[d]["ndcg"], after_domains[d]["ndcg"])
                     for d in after_domains), key=lambda r: -r[1])

    fig, ax = plt.subplots(figsize=(9.2, 5.0))
    values = [r[1] for r in deltas]
    bars = ax.barh([r[0] for r in deltas], values,
                   color=[TEAL if v >= 0 else ORANGE for v in values],
                   height=0.62)
    ax.invert_yaxis()
    ax.axvline(0, color=AXIS, linewidth=0.9)
    span = max(abs(v) for v in values) or 1
    ax.set_xlim(min(values) - span * 0.42, max(values) + span * 0.42)
    for bar, (domain, delta, was, now) in zip(bars, deltas):
        at_right = delta >= 0
        ax.text(delta + span * (0.03 if at_right else -0.03),
                bar.get_y() + bar.get_height() / 2,
                f"{delta:+.4f}   {was:.3f} -> {now:.3f}",
                va="center", ha="left" if at_right else "right",
                fontsize=8.5, color=INK_SOFT)
    won = sum(1 for v in values if v > 0)
    finish(ax, f"Xep hang lai giup o {won}/{len(values)} nhom",
           subtitle=f"{RERANK_MODEL}, thay doi nDCG@10 theo tung nhom",
           xlabel="thay doi nDCG@10", grid_axis="x")
    plt.show()

### Điều cần đọc ở biểu đồ này

Nếu có nhóm bị âm, đó là tín hiệu đáng chú ý chứ không phải nhiễu. Cross-encoder
được huấn luyện trên dữ liệu web tiếng Anh chung, còn các nhóm ở đây là diễn đàn
chuyên ngành hẹp. Một nhóm bị tụt nghĩa là mô hình hiểu sai loại văn bản đó.

Hai hướng xử lý, nếu chuyện đó xảy ra:

- Gộp điểm của tầng một với điểm của bộ xếp hạng lại thay vì thay thế hẳn, để
  nhóm bị tụt vẫn giữ được phần lớn thứ tự cũ.
- Dùng bộ xếp hạng lại khác. Đổi mô hình chỉ cần xuất một gói việc mới với
  `JOB_ID` khác, không phải chạy lại tầng một.

In [ ]:
# --- Record the outcome ---------------------------------------------------
if rerank_digests:
    rerank_verdict = {
        "job_id": JOB_ID,
        "model": RERANK_MODEL,
        "first_stage": stage1["name"],
        "depth": stage1["retrieve"]["depth"],
        "pairs_scored": job["pairs"],
        "before": {"ndcg": before_score, "ceiling": ceiling},
        "after": {name: P.score(d)["macro"]["ndcg"]
                  for name, d in rerank_digests.items()},
        "best": best_name,
        "headroom_collected": collected,
        "significance": rerank_significance,
        "per_domain_delta": {d: delta for d, delta, _, _ in deltas},
        "buckets": {k: len(v) for k, v in buckets_deep.items()},
    }
    (CACHE / "rerank_summary.json").write_text(
        json.dumps(rerank_verdict, indent=1), encoding="utf-8")
    print(f"Saved {CACHE / 'rerank_summary.json'}")
    print()
    P.table()

---

## 5. Kết luận

### Hai việc, hai phần thưởng

| Việc | Đo bằng | Đọc ở mục |
| --- | --- | --- |
| Lấy sâu hơn | `ceiling` và nhóm "ngoài tầm với" | 1, 2 |
| Xếp hạng lại | `nDCG@10` và phần trần giành được | 4 |

Tách hai việc ra là chủ ý. Nếu chỉ chạy cả hai rồi nhìn điểm cuối thì không
biết phần nào đến từ đâu, và lần sau không biết nên đầu tư vào đâu.

### Gói việc GPU là một phần của kết quả

Thư mục `gpu_jobs/rerank_01/` ghi lại đúng danh sách ứng viên đã gửi đi. Người
khác chạy lại đúng gói đó với đúng mô hình đó sẽ ra đúng số đó. Mã băm của hệ
thống có chứa `JOB_ID`, nên một hệ thống dùng gói khác là một mã băm khác và
không lẫn vào nhau được.

### Bước tiếp theo

Nhóm "ngoài tầm với" còn lại là phần `bm25` không với tới. Notebook `07c` thử
một cách truy xuất khác hẳn để xem có với tới được không.